In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys

sys.path.insert(0, '/scratch/git/tunix')

import jax
import jax.numpy as jnp
from transformers import AutoTokenizer, AutoProcessor
from tunix.generate import sampler
from tunix.models.gemma4 import model
from tunix.models.gemma4 import params_safetensors


MESH = [(4, 2), ("fsdp", "tp")]
mesh = jax.make_mesh(
    *MESH, axis_types=(jax.sharding.AxisType.Auto,) * len(MESH[0])
)

# Download HF safetensors to your local dir.
# e.g. hf download google/gemma-4-31B-it --local-dir <your local path>
E2B_PATH = "/scratch/models/Gemma4-E2B-it"
E4B_PATH = "/scratch/models/Gemma4-E4B-it"
MOE_26B_PATH = "/scratch/models/Gemma4-26B-A4B-it"
DENSE_31B_PATH = "/scratch/models/Gemma4-31B-it"

version = "e2b"
if version == "e2b":
  tokenizer = AutoTokenizer.from_pretrained(E2B_PATH)
  config = model.ModelConfig.gemma4_e2b()
  config.vision_encoder.output_length = 70
  config.use_flash_attention = True
  config.remat_config = model.RematConfig.BLOCK
  model = params_safetensors.create_model_from_safe_tensors(
      E2B_PATH, config, mesh, dtype=jnp.bfloat16, text_only=True
  )
  cache_config = sampler.CacheConfig(
      cache_size=8192, num_layers=35, num_kv_heads=1, head_dim=512
  )
  processor = AutoProcessor.from_pretrained(E2B_PATH)
elif version == "e4b":
  tokenizer = AutoTokenizer.from_pretrained(E4B_PATH)
  config = model.ModelConfig.gemma4_e4b()
  config.vision_encoder.output_length = 70
  config.use_flash_attention = True
  config.remat_config = model.RematConfig.DECODER
  model = params_safetensors.create_model_from_safe_tensors(
      E4B_PATH, config, mesh, dtype=jnp.bfloat16, text_only=True
  )
  cache_config = sampler.CacheConfig(
      cache_size=2048, num_layers=42, num_kv_heads=2, head_dim=512
  )
  processor = AutoProcessor.from_pretrained(E4B_PATH)
elif version == "moe_26b":
  tokenizer = AutoTokenizer.from_pretrained(MOE_26B_PATH)
  config = model.ModelConfig.gemma4_26b_a4b()
  model = params_safetensors.create_model_from_safe_tensors(
      MOE_26B_PATH, config, mesh, dtype=jnp.bfloat16, text_only=False
  )
  config.use_flash_attention = True
  cache_config = sampler.CacheConfig(
      cache_size=2048, num_layers=30, num_kv_heads=8, head_dim=512
  )
  processor = AutoProcessor.from_pretrained(MOE_26B_PATH)
elif version == "31b":
  tokenizer = AutoTokenizer.from_pretrained(DENSE_31B_PATH)
  config = model.ModelConfig.gemma4_31b()
  config.use_flash_attention = True
#   config.remat_config = model.RematConfig.DECODER
  model = params_safetensors.create_model_from_safe_tensors(
      DENSE_31B_PATH, config, mesh, dtype=jnp.bfloat16
  )
  cache_config = sampler.CacheConfig(
      cache_size=2048, num_layers=60, num_kv_heads=16, head_dim=512
  )
  processor = AutoProcessor.from_pretrained(DENSE_31B_PATH)
else:
  raise ValueError(f"Unknown version: {version}")

In [3]:
import logging
import sys

# Remove existing handlers to prevent duplicate logs or conflicts
for handler in logging.root.handlers[:]:
    logging.root.removeHandler(handler)

logging.basicConfig(
    stream=sys.stdout,  # Direct logs to standard output (notebook cell)
    level=logging.INFO, # Set the minimum level to INFO
    format="%(asctime)s - %(levelname)s - %(message)s", # Optional: customize the format
    datefmt="%Y-%m-%d %H:%M:%S" # Optional: customize the date format
)
from tunix.sft import utils as sft_utils

sft_utils.show_hbm_usage()

2026-07-23 04:37:43 - INFO -  - Pathways not available. Using default HBM stats collector
2026-07-23 04:37:43 - INFO - Using 1.1 GiB / 31.2 GiB (0.03493805682473698) on TPU_0(process=0,(0,0,0,0))
2026-07-23 04:37:43 - INFO - Using 1.1 GiB / 31.2 GiB (0.0348878186052332) on TPU_1(process=0,(1,0,0,0))
2026-07-23 04:37:43 - INFO - Using 1.1 GiB / 31.2 GiB (0.0348878186052332) on TPU_2(process=0,(0,1,0,0))
2026-07-23 04:37:43 - INFO - Using 1.1 GiB / 31.2 GiB (0.0348878186052332) on TPU_3(process=0,(1,1,0,0))
2026-07-23 04:37:43 - INFO - Using 1.1 GiB / 31.2 GiB (0.0348878186052332) on TPU_4(process=0,(0,2,0,0))
2026-07-23 04:37:43 - INFO - Using 1.1 GiB / 31.2 GiB (0.0348878186052332) on TPU_5(process=0,(1,2,0,0))
2026-07-23 04:37:43 - INFO - Using 1.1 GiB / 31.2 GiB (0.0348878186052332) on TPU_6(process=0,(0,3,0,0))
2026-07-23 04:37:43 - INFO - Using 1.1 GiB / 31.2 GiB (0.0348878186052332) on TPU_7(process=0,(1,3,0,0))


In [5]:
from tunix.sft import peft_trainer
import optax
from tunix.sft import utils
from tunix.examples.data import translation_dataset as data_lib
from tunix.generate import tokenizer_adapter as tokenizer_lib

# train_ds, validation_ds = data_lib.create_datasets(
#     dataset_name='mtnt/en-fr',
#     # Uncomment the line below to use a Hugging Face dataset.
#     # Note that this requires upgrading the 'datasets' package and restarting
#     # the Colab runtime if you are using it.
#     # dataset_name='Helsinki-NLP/opus-100',
#     global_batch_size=4,
#     max_target_length=1024,
#     num_train_epochs=1,
#     tokenizer=tokenizer_lib.TokenizerAdapter(tokenizer),
# )

def gen_model_input_fn(x: peft_trainer.TrainingInput):
  pad_mask = x.input_tokens != 0
  positions = utils.build_positions_from_mask(pad_mask)
  attention_mask = utils.make_causal_attn_mask(pad_mask)
  return {
      'input_tokens': x.input_tokens,
      'input_mask': x.input_mask,
      'positions': positions,
      'attention_mask': attention_mask,
  }


training_config = peft_trainer.TrainingConfig(
    eval_every_n_steps=10,
    max_steps=3,
)

trainer = peft_trainer.PeftTrainer(
    model, optax.adamw(1e-5), training_config
).with_gen_model_input_fn(gen_model_input_fn)

2026-07-23 04:37:43 - INFO - PyTorch version 2.12.0+cpu available.
2026-07-23 04:37:43 - INFO - Polars version 1.37.1 available.
2026-07-23 04:37:43 - INFO - TensorFlow version 2.21.0 available.
2026-07-23 04:37:43 - INFO - JAX version 0.9.2 available.


In [ ]:
from tunix.sft.peft_trainer import TrainingInput

ds = []

for _ in range(10):
    ds.append(TrainingInput(
        input_tokens=jnp.ones((4, 2048), dtype=jnp.int32),
        input_mask=jnp.ones((4, 2048), dtype=jnp.int32),
    ))

# with mesh:  
with mesh, jax.profiler.trace(log_dir="gs://tsbao-dev/optimizer_sharding"):
    trainer.train(ds, None)

2026-07-23 04:37:45 - INFO - Training with mesh: Mesh('fsdp': 4, 'tp': 2, axis_types=(Auto, Auto))
2026-07-23 04:37:45 - INFO - Compiled train_step cache size: 0


Training:   0%|          | 0/3 [00:00<?, ?step/s]